# Příklad 1 — dvojný integrál + bisekce

Najdi `p` takové, aby platilo:

$$\int_0^p \left( \int_0^{\pi} e^{-\alpha x} \cos(x) \, dx \right) d\alpha = 0.85$$

**Postup:**
1. Vnitřní integrál — Simpson
2. Vnější integrál — Simpson
3. Najdi `p` — bisekce

**Ověř** dosazením zpět.

In [14]:
import math
def simpson_rule(f, a, b, n):
    """
    Numerický výpočet integrálu pomocí Simpsonova pravidla.
    
    :param f: Integrovaná funkce
    :param a: Dolní mez
    :param b: Horní mez
    :param n: Počet dělení (musí být sudé a >= 2)
    :return: Přibližná hodnota integrálu
    """
    # Simpson vyžaduje sudý počet intervalů (n+1 bodů)
    if n < 2 or n % 2 != 0:
        print("Chyba: n musí být sudé a >= 2.")
        return None

    h = (b - a) / n
    # Začneme součtem krajních hodnot f(a) + f(b)
    total_sum = f(a) + f(b)

    for i in range(1, n):
        x = a + i * h
        # Body v lichých pozicích se násobí 4, v sudých 2
        if i % 2 == 0:
            total_sum += 2 * f(x)
        else:
            total_sum += 4 * f(x)

    # Konečný vzorec: (h/3) * (f(a) + 4*f(x1) + 2*f(x2) + 4*f(x3) + ... + f(b))
    approximation = (h / 3) * total_sum
    return approximation



def bisection(f, a, b, tol, max_iter, verbose=False):
    """
    Numerické hledání kořene funkce pomocí metody půlení intervalu.

    :param f: Funkce, jejíž kořen hledáme
    :param a: Levá mez intervalu
    :param b: Pravá mez intervalu
    :param tol: Tolerance (přesnost)
    :param max_iter: Maximální počet iterací
    :param verbose: Vypisovat průběh iterací (default False)
    :return: Kořen funkce nebo None, pokud metoda selže
    """
    fa = f(a)
    fb = f(b)

    if fa * fb > 0:
        print("Chyba: Žádná změna znaménka, kořen v intervalu nelze zaručit.")
        return None

    c = a
    for i in range(max_iter):
        c = (a + b) / 2
        fc = f(c)

        if verbose:
            print(f"Iterace {i+1:3d}: c = {c:.10f}, f(c) = {fc:.4e}, délka intervalu = {(b-a)/2:.4e}")

        if abs(fc) < tol or (b - a) / 2 < tol:
            if verbose:
                print(f"Konvergoval v iteraci {i+1}.")
            return c

        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc

    print("Chyba: Metoda neskonvergovala.")
    return None

# Příklad použití:
# root = bisection(lambda x: x**2 - 4, 0, 5, 1e-6, 100)
# print(f"Kořen je: {root}")


def vnitrni(alpha):

    vnitr_fun = lambda x: math.e**(-alpha*x)*math.cos(x)
    return simpson_rule(vnitr_fun,0,math.pi,100)


def vnejsi(p):
    return simpson_rule(vnitrni,0,p,100)


def vysledek(a,b):
    target = 0.85
    fukce = lambda p: vnejsi(p) - target
    p = bisection(fukce,a,b,1e-8,100,verbose=True)
    vnejsi(p)
    print(f"p = {p:.10f}")
    print(f"ověření: vnejsi(p) = {vnejsi(p):.6f}")


#for p in range(-2, 20):
#    print(f"p={p}, vnejsi(p)-0.85 = {vnejsi(p) - 0.85}")

vysledek(0,2)


Iterace   1: c = 1.0000000000, f(c) = -4.3631e-01, délka intervalu = 1.0000e+00
Iterace   2: c = 1.5000000000, f(c) = -1.8822e-01, délka intervalu = 5.0000e-01
Iterace   3: c = 1.7500000000, f(c) = -7.5946e-02, délka intervalu = 2.5000e-01
Iterace   4: c = 1.8750000000, f(c) = -2.2894e-02, délka intervalu = 1.2500e-01
Iterace   5: c = 1.9375000000, f(c) = 2.8817e-03, délka intervalu = 6.2500e-02
Iterace   6: c = 1.9062500000, f(c) = -9.9447e-03, délka intervalu = 3.1250e-02
Iterace   7: c = 1.9218750000, f(c) = -3.5162e-03, délka intervalu = 1.5625e-02
Iterace   8: c = 1.9296875000, f(c) = -3.1340e-04, délka intervalu = 7.8125e-03
Iterace   9: c = 1.9335937500, f(c) = 1.2851e-03, délka intervalu = 3.9062e-03
Iterace  10: c = 1.9316406250, f(c) = 4.8610e-04, délka intervalu = 1.9531e-03
Iterace  11: c = 1.9306640625, f(c) = 8.6412e-05, délka intervalu = 9.7656e-04
Iterace  12: c = 1.9301757812, f(c) = -1.1348e-04, délka intervalu = 4.8828e-04
Iterace  13: c = 1.9304199219, f(c) = -1.352

# Příklad 2 — shooting method

Máš ODE:

$$y'(x) - y + x = 0$$

a podmínku:

$$\int_0^1 y(x) \, dx = 2$$

Najdi `y(0)` a vykresli řešení na intervalu `[0, 1]`.



In [18]:


import math

def heun(f, x0, y0, h, n):
    """
    Řešení ODR y' = f(x, y) Heunovou metodou (zpřesněná Eulerova metoda, RK2).

    Heun = prediktor (Euler krok) + korektor (průměr sklonů).
    Vzorec:
        k1 = f(x, y)
        k2 = f(x + h, y + h*k1)
        y_new = y + (h/2) * (k1 + k2)

    :param f: Funkce f(x, y) definující y' = f(x, y)
    :param x0: Počáteční hodnota x
    :param y0: Počáteční hodnota y
    :param h: Krok integrace
    :param n: Počet kroků
    :return: Seznam bodů [(x0,y0), (x1,y1), ...] nebo None při chybě
    """
    if n < 1:
        print("Chyba: Počet kroků n musí být >= 1.")
        return None

    points = [(x0, y0)]
    x = x0
    y = y0

    for i in range(n):
        k1 = f(x, y)
        k2 = f(x + h, y + h * k1)

        if any(math.isnan(k) or math.isinf(k) for k in (k1, k2)):
            print("Chyba: Metoda divergovala (NaN/Inf).")
            return None

        y_new = y + (h / 2) * (k1 + k2)
        x_new = x + h

        if math.isnan(y_new) or math.isinf(y_new):
            print("Chyba: Metoda divergovala (NaN/Inf).")
            return None

        points.append((x_new, y_new))
        x, y = x_new, y_new

    return points



def prepisODE(x,y):
    return y-x






In [ ]:
import math
import sys
import os




# ============================================================
# 1. PŘEPIS ODE ZE ZADÁNÍ
#    y' = f(x, y)  →  return <vzorec ze zadání>
# ============================================================
def f_ode(x, y):
    return y - x          # <-- ZDE DOSAĎ VZOREC ZE ZADÁNÍ


# ============================================================
# 2. RESIDUAL FUNKCE
#    Spočítá integrál y(x) na [x0, xk] pro dané y(0) = s
#    a vrátí rozdíl od cílové hodnoty TARGET
# ============================================================
TARGET = 2                # <-- ZDE DOSAĎ CÍLOVOU HODNOTU ZE ZADÁNÍ
X0 = 0                    # počáteční bod intervalu
XK = 1                    # koncový bod intervalu
H = 0.1                   # krok integrace
N = int((XK - X0) / H)    # počet kroků


def residual(s):
    """
    :param s: střelecká počáteční podmínka y(x0) = s
    :return: integral y(x) dx - TARGET
    """
    #body = runge_kutta_4(f_ode, X0, s, H, N)
    body = heun(f_ode, X0, s, H, N)

    ys = [y for x, y in body]

    # lichoběžníkové pravidlo
    integral = H * (ys[0]/2 + sum(ys[1:-1]) + ys[-1]/2)

    # Simpson (přesnější, N musí být sudé)
    # integral = (H/3) * (ys[0] + 4*sum(ys[1::2]) + 2*sum(ys[2:-1:2]) + ys[-1])

    return integral - TARGET


# ============================================================
# 3. BISEKCE — najde s takové aby residual(s) = 0
#    Uprav interval [a, b] podle zadání
# ============================================================
A = -10                   # <-- ZDE UPRAV interval pro bisekci
B = 10                    # <-- ZDE UPRAV interval pro bisekci

s = bisection(residual, A, B, 1e-8, 100)

print(f"y(0) = {s:.10f}")
print(f"ověření: integral = {residual(s) + TARGET:.6f}  (má být {TARGET})")

y(0) = 1.2910086475
ověření: integral = 2.000000  (má být 2)
